# Arrhenius kinetics with correlated parameters: ORB and SEM designs for ill-conditioned problems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/tutorials/04_design_precision/b_ill_conditioned/tutorial.ipynb)

**MBDoE for ill-conditioned systems** (`mbdoe_pp` with `ORB` and `SEM`).

Arrhenius parameters are notoriously correlated: a larger rate constant and a larger activation
energy can produce almost the same curve. The Fisher matrix is then **ill-conditioned**: some
directions in parameter space are well known, others almost not at all. D-optimal designs tend to
add information where there is already plenty. Two criteria target the weak directions:

* **ORB**, orthogonal residual balancing: raises the information in the poorly known directions;
* **SEM**, spectral entropy maximisation: spreads the information evenly over the eigenvalues.

You will start from two isothermal batches, look at the condition of the problem, design with
D, ORB and SEM, and run the SEM design.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s); does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    for f in ['model.py']:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/tutorials/04_design_precision/b_ill_conditioned/' + f, f)

### The process

Batch synthesis of a quaternary ammonium salt (an intermediate of an HIV drug):

$$\mathrm{SM + TMA \rightleftharpoons QS1Cl \qquad QS1Cl \rightarrow ClDMI + MeCl}$$

The product QS1Cl forms reversibly and slowly degrades into the impurity ClDMI. Rate and
equilibrium constants follow Arrhenius / van 't Hoff laws around 296.15 K, so the model has six
parameters: three constants at the reference temperature and three energies. Measured: starting
material `y1`, product `y2`, impurity `y3`. Inputs: the temperature profile `u1` and the initial
concentrations `y10`, `y20`.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

from middoe.krnl_expera import expera
from middoe.iden_parmest import parmest
from middoe.iden_uncert import uncert
from middoe.log_utils import save_rounds
from middoe.des_pp import mbdoe_pp

QUICK = False   # True: tiny budgets and fewer designs, a check that everything runs (minutes), not a result.
HERE = Path.cwd()
# Start clean: expera() ADDS sheets to an existing data.xlsx, so remove what an earlier run wrote.
for old in ['data.xlsx', 'data_plots', 'modelling', 'confidence', 'post_processing', 'estimability',
            'meshindep_plots', 'design_plots', 'iden_results.jac']:
    p = HERE / old
    shutil.rmtree(p) if p.is_dir() else p.unlink(missing_ok=True)
np.random.seed(0)                                # same noise at every run

In [ ]:
theta_true  = [50000, 75000, 0.4116, 111900, 9905, 30000]   # the in-silico 'real' process
theta_guess = [100000, 100000, 1, 100000, 100, 10000]       # starting values
theta_min   = [10000, 0, 0.1, 50000, 10, 10000]             # bounds
theta_max   = [1000000, 200000, 10, 200000, 10000, 200000]
NAMES = ['kf_ref', 'Ef', 'kfs_ref', 'Efs', 'K_ref', 'dH']   # theta1 ... theta6 of model.py

sample = {'init': 0, 'meas': True,  # every output: measured, 17 synchronised samples at least 0.3 h apart,
          'sp': 17, 'offt': 0.3,    # always at 0 and 16 h; noise sd 0.005 mol/L
          'samp_s': 1, 'samp_f': [0, 16], 'unc': 0.005}

system = {
    'tvi': {'u1': {                       # reactor temperature [K]
        'stps': 6, 'cvp': 'LPF', 'const': 'inc',   # 6 levels joined by ramps, rising only
        'min': 296.15, 'max': 306.15,
        'offl': 0.01, 'offt': 0.3,                 # min level change [K], min time between switches [h]
    }},
    'tvo': {'y1': dict(sample), 'y2': dict(sample), 'y3': dict(sample)},  # SM, product, impurity [mol/L]
    'tii': {'y10': {'min': 0.3, 'max': 1},         # initial starting material [mol/L]
            'y20': {'min': 0.19, 'max': 1}},       # initial trimethylamine [mol/L]
    'tio': {},
    't_s': [0, 16],      # 16-h batch
    't_r': 0.1 if QUICK else 0.02,   # time grid [h] (coarser in a quick check)
    't_d': 0.3,          # dead time at both ends [h] (designs only)
}

models = {
    'can_m': ['M'],
    'krt': {'M': 'pys'},                         # a Python file with solve_model(t, y0, tii, tvi, theta)
    'src': {'M': str(Path('model.py').resolve())},
    'theta': {'M': list(theta_guess)},
    't_l': {'M': theta_min},
    't_u': {'M': theta_max},
    'mutation': {'M': [True] * 6},               # True = estimated, False = held at its value
    'theta_names': {'M': NAMES},
}

insilicos = {'tr_m': 'M', 'theta': theta_true, 'errt': 'abs',
             'prels': {'1': {'u1': 296.15, 'y10': 0.366, 'y20': 0.19},
                       '2': {'u1': 296.15, 'y10': 0.65, 'y20': 0.595}}}   # both at 296 K

iden_opt = {'meth': 'LMBFGS', 'ms': not QUICK, 'ob': 'WLS', 'maxit': 60 if QUICK else 500, 'tol': 1e-8,
            'var-cov': 'M', 'sens_m': 'central', 'eps': 1e-5,
            'c_plt': False, 'f_plt': True, 'plt_s': True, 'log': False}


def conditioning(unc, model='M'):
    """Correlations and condition number of the parameter covariance (scaled to unit variance)."""
    V = np.asarray(unc['results'][model]['V_matrix'], float)
    if not np.all(np.isfinite(V)) or np.any(np.diag(V) <= 0):
        print('the covariance is not finite: these data cannot identify all the parameters')
        return None
    V = (V + V.T) / 2                            # symmetric part (guards against round-off)
    s = np.sqrt(np.diag(V)); C = V / np.outer(s, s)
    ev = np.linalg.eigvalsh(C)
    act = np.asarray(unc['results'][model]['optimization_result']['activeparams'], bool)
    names = [n for n, a in zip(NAMES, act) if a]
    print(f'condition number of the correlation matrix: {ev.max() / ev.min():.3g}')
    return pd.DataFrame(C, index=names, columns=names).round(2)

### Two isothermal batches

In [ ]:
round_data = {}
expera(system, models, insilicos, {}, expr=1)
expera(system, models, insilicos, {}, expr=2)
resultpr = parmest(system, models, iden_opt, case='strov')
unc = uncert(resultpr, system, models, iden_opt)
save_rounds(1, unc['results'], 'isothermal', round_data, models, iden_opt, unc['obs'], system)
conditioning(unc)

Correlations close to ±1 and a condition number of many orders of magnitude: the energies
are not identifiable from isothermal data (tutorial 2a showed why).

### Three designs

In [ ]:
def des_settings(criterion):
    return {'pp_ob': criterion, 'meth': 'PS', 'eps': 1e-5, 'priors': True, 'plt': True,
            'itr': {'maxpp': 15 if QUICK else 100, 'tolpp': 1, 'pps': 30}}

designs = {}
for crit in (['SEM'] if QUICK else ['D', 'ORB', 'SEM']):
    print(f'---------------- {crit}')
    designs[crit] = mbdoe_pp(des_settings(crit), system, models, round=2,
                             num_parallel_runs=1 if QUICK else 4)
pd.DataFrame({c: {'y10': d['tii'].get('y10'), 'y20': d['tii'].get('y20'),
                  'temperature levels [K]': np.round(np.unique(np.asarray(d['tvi']['u1'], float)), 1)[:6]}
              for c, d in designs.items()}).T

All three raise the temperature (the only way to see the energies), but ORB and SEM do so
to balance the spectrum rather than to grow the determinant.

### Run the SEM design and refit

In [ ]:
d = designs['SEM']
expera(system, models, insilicos, d, expr=3, swps=d['swps'])
resultpr = parmest(system, models, iden_opt, case='strov')
unc = uncert(resultpr, system, models, iden_opt)
save_rounds(2, unc['results'], 'MBDoE-PP (SEM)', round_data, models, iden_opt, unc['obs'], system)
conditioning(unc)

**Try:** run the ORB or the D design instead (restart the kernel and run all, changing the
key) and compare the condition numbers. With strongly correlated parameters the case study of the
ORB/SEM paper reports the largest gains for the poorest-known parameter.